<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Introduction to Multiple Linear Regression - Solution

---

### About
An introduction to Linear Regression: what it is, how it works, and how to run a model using `scikit-learn`.

### Learning Objectives
- Describe modeling.
- Calculate mean squared error.
- State the assumptions of a linear regression model.
- Be able to interpret the coefficients of a linear regression model.
- Identify the difference between simple and multiple linear regression.
- Fit, generate predictions from, and evaluate a linear regression model in `sklearn`, and maybe in `statsmodels`.

### Notebook Guide

Part II. **Multiple Linear Regression**
- Introduction to MLR Model (see slides)
- Fitting an MLR model in `sklearn`
- Assumptions of MLR Model
- NOW YOU TRY
- Additional, not-really-BONUS topics:
    1. Multicollinearity
    1. The Curse of Dimensionality
    1. Fitting an MLR model in `statsmodels`
    1. Does the Scale of the Features Matter?
- Conclusions and Takeaways

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn import metrics

# Part II: Multiple Linear Regression

When you have more than 1 predictor variable you are doing _multiple linear regression_.

## Bikeshare Data 🚴‍♂️
Data adapted from source: [here](https://archive.ics.uci.edu/ml/datasets/bike+sharing+dataset)


Multiple Linear Regression (MLR) with Categorical Features

#### Goal: predict the number of riders (count), based on the features.

In [ ]:
bike = pd.read_csv("../data/bikeshare.csv")
print(bike.shape)
bike.head()

### Some preprocessing to clean things up:
1. Rename "count" to "riders"
2. Drop "casual" and "registered"
3. Cast "datetime" to actually be a datetime
4. Drop rows where weather is 4

In [ ]:
# Some preprocessing:
# 1 - Rename "count" to "riders"
# 2 - Drop "casual" and "registered"
# 3 - Cast "datetime" to actually be a datetime
# 4 - Drop only row where weather == 4

bike.rename(columns={"count": "riders"}, inplace=True)
bike.drop(columns=["casual", "registered"], inplace=True)
bike["datetime"] = pd.to_datetime(bike["datetime"])
bike = bike[bike["weather"] != 4]

In [ ]:
# Season vs Month
pd.crosstab(bike["season"], bike["datetime"].dt.month)

### Ridership model
Let's try to predict the number of riders based on temperature and what season it is. We'll have to **one-hot encode** the season column!

### Step 1: Assemble our predictor variables (X) and our target (y) 

In [ ]:
# Dummify "season"
X = bike[["season", "temp"]]
X = pd.get_dummies(columns=["season"], drop_first=True, data=X, dtype=int)

y = bike["riders"]

In [ ]:
X.head()

### Step 2: Instantiate the model

In [ ]:
model = LinearRegression()

### Step 3: Fit the model

In [ ]:
# Fit model
model.fit(X, y)

### Step 4: Check out and interpret our coefficients

In [ ]:
# Coefficients
model.coef_

In [ ]:
set(zip(X.columns, model.coef_))

#### Interpretation of coefficients
Now that we have multiple variables, our interpretation are **holding all other variables constant**. 
> For every 1 unit increase in $x_i$, we expect $y$ to increase by $\beta_i$, holding all else constant.

Furthermore, each interpretation of a dummy variable is **in relation to the baseline** (in this case, Winter).

**Interpretation of $\hat{\beta}_3$ (`season_3` - Summer):**
Holding the temperature constant, the effect of it being Summer contributes to 105 fewer riders per hour relative to Winter.

<details><summary>Interpret the coefficient for temp.</summary>
Holding all else constant, for every one-degree increase in temperature, we expect about 14 more riders per hour.
</details>

<details><summary>Interpret the coefficient for season_4 (Fall).</summary>
Holding the temperature constant, the effect of it being Fall contributes to 26 more riders per hour relative to Winter.
</details>

### Step 5: Make predictions

In [ ]:
y_preds = model.predict(X)
y_preds

### Step 6: Evaluate the model

In [ ]:
# R2:
metrics.r2_score(y, y_preds)

In [ ]:
# MSE:
metrics.mean_squared_error(y, y_preds)

In [ ]:
# RMSE:
metrics.root_mean_squared_error(y, y_preds)

<details><summary>How is our model?</summary>
Overall, this model isn't great. Only about 20% of the variation in ridership can be explained by the model.
</details>

So our model is pretty bad. How much should we trust our interpretation for the coefficients?
> Do our LINE assumptions hold?

When working with multiple linear regression, we add a secret 5th "assumption" to our LINE assumptions that is usually violated at least a little.

### LINEM
- L - Linear relationship
- I - Independent errors
- N - Normally distributed errors
- E - Equal variance of errors (homoscedasticity)
- M - No Multicollinearity/Independence of Predictors

In [ ]:
# L:
plt.scatter(X['temp'], y);

In [ ]:
sns.pairplot(pd.merge(X, y, left_index = True, right_index = True));

In [ ]:
# I:
# assumed
# although this is probably violated since this is time-series data

In [ ]:
# N:
resids = y - y_preds
plt.hist(resids, bins=50);

In [ ]:
# E:
plt.scatter(y_preds, resids, s=1)
plt.axhline(0, color="orange");

In [ ]:
# M/I:
# Honestly temperature might be a little correlated with season, but not enough to be worried
X.corr()

In [ ]:
mask = np.zeros_like(X.corr())
mask[np.triu_indices_from(mask)] = True

plt.figure(figsize=(10, 10))
sns.heatmap(
    X.corr(),
    cmap='coolwarm',
    annot=True,
    mask = mask,
    vmin = -1,
    vmax = 1
);

Yikes! We should probably not say anything very strong about the coefficients in our interpretation. 

# NOW YOU TRY

Let's make a better model.

Now your turn: Fit a model that uses _all_ x-variables (excluding `atemp`) as well as the **the hour of day**. One-hot encode columns accordingly, and check the assumptions.

In [ ]:
bike.head()

In [ ]:
bike["hour"] = bike["datetime"].dt.hour

xvars = [
    "season", "holiday", "workingday",
    "weather", "temp", "humidity",
    "windspeed", "hour"
]

X = bike[xvars]
X = pd.get_dummies(columns=["season", "weather", "hour"], data=X, drop_first=True)

In [ ]:
X.head()

In [ ]:
y = bike['riders']

In [ ]:
model_big = LinearRegression()
model_big.fit(X, y)

In [ ]:
model_big.score(X, y)

In [ ]:
y_preds = model_big.predict(X)

In [ ]:
# N:
resids = y - y_preds
plt.hist(resids, bins=50);

In [ ]:
# E

plt.scatter(y_preds, resids, s=1)
plt.axhline(0, color="orange");

# Additional, not-really-BONUS topics

## ANRBT #1: Multicollinearity
Multicollinearity occurs when you have two or more $x$-variables that are correlated with each other. This phenomenon doesn't affect the quality of the model fit, but it completely destroys model inference (i.e., confidence intervals and hypothesis tests). In addition, it can also contribute to a **rank-deficient matrix**, meaning the model can't be fit at all!

In [ ]:
np.corrcoef(bike.temp, bike.atemp)

In [ ]:
bike[['temp', 'atemp']].corr()

## ANRBT #2: The Curse of Dimensionality
It turns out that the more variables we add to our model, the better it is. Actually, it doesn't even matter what this variable was, that will always be true. Even if it's completely irrelevant. That is, if we added the following variables to our model:

* Is there a sunspot on this day?
* The closing stock price of Apple that day.
* Whether or not Guy Fieri had pancakes for breakfast that day.
* A column of completely random standard normal variables

Our model is **guaranteed** to get better. But it isn't _really_ a better model, right? It turns out, we're modeling **noise**, not **signal**. We'll talk more about this phenomenon later.

## ANRBT #3: The `statsmodels` Library
`scikit-Learn` does not hold the monopoly in doing linear regression in Python. In fact, you may even like this better!

In [ ]:
import statsmodels.api as sm

elec = pd.read_csv("../data/elecdemand.csv")

X = elec[["temp", "workday"]]
X = sm.add_constant(X)
y = elec["demand"]
ols = sm.OLS(y, X).fit()

In [ ]:
ols.summary()

## ANRBT #4: Does the Scale of the Features Matter?

Let's say that temperature was measured in Fahrenheit, rather than Celsius. How would that affect the model?

In [ ]:
# Create a new column for Fahrenheit temperature.
bike['temp_F'] = bike['temp'] * 1.8 + 32
bike.head()

#### Rebuild the `LinearRegression` from above using the `temp_F` features instead.

In [ ]:
# Create X and y.
feature_cols = ['temp_F']
X = bike[feature_cols]
y = bike['riders']

# Instantiate and fit.
linreg_scale = LinearRegression()
linreg_scale.fit(X, y)

# Print the coefficients.
print(linreg_scale.intercept_)
print(linreg_scale.coef_)

#### Convert 25 degrees Celsius to Fahrenheit.

In [ ]:
25 * 1.8 + 32

#### Predict rentals for 77 degrees Fahrenheit.

In [ ]:
linreg_scale.predict(np.array([[77]]))

#### Compare with the prediction for 25 degrees Celsius.

In [ ]:
# Re-set up the model from earlier

feature_cols = ['temp']
X = bike[feature_cols]
y = bike['riders']
lr = LinearRegression()
lr.fit(X, y)

# Examine the prediction
lr.predict(np.array([[25]]))

**Conclusion:** The scale of the features is irrelevant for linear regression models. When changing the scale, we simply change our interpretation of the coefficients and the intercept.

# Conclusions and Takeaways:
* Machine learning, aka statistical modeling, is an important and vast topic we'll spend much of our course exploring. It comes in many varieties for different situations.
* Linear regression is a **supervised "white-box" regression model**.
* As far as machine learning goes, OLS is very simple, yet still powerful.
* One of its more enticing feature is that its **coefficients are interpretable!**
* In the midst of all this, we learned how to handle dummy variables, something important for all of the models we'll learn about.